# 03e · M/M/c와 유한 용량 (M/M/c Queues and Finite Capacity)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §4.5 (M/M/s, 유한 대기실) · Ross 12e §8.3 (유한 용량·출생-사망 대기열), §8.9.1 (Erlang 손실 시스템), §8.9.2 (M/M/k) · Norris §5.2.2 (M/M/s), §5.2.3 (전화 교환기 M/M/s/s) · Lawler 2e §3.3 (출생-사망 과정) |
| 선수 노트북 | 03d (M/M/1, 리틀의 법칙, PASTA, 배치 평균 SE) |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **서버가 여러 대이거나 자리가 유한해도 대기행렬은 여전히 출생-사망 CTMC이고, 03c의 곱 형태 하나에서 Erlang C(대기 확률)와 Erlang B(차단 확률)가 모두 나온다.** 설계 질문 — "빠른 서버 1대가 나은가, 느린 서버 2대가 나은가?" — 의 답은 지표($W$인가 $W_q$인가)에 따라 **반대**가 된다는 것이 이 노트북의 교훈이다. 시뮬레이션은 03d의 손님 단위 방식에 '가장 먼저 비는 서버' 힙을 얹는다.

## 0. 준비 (Setup)

In [ ]:
import os
import heapq
from math import factorial
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import mmc_metrics, birth_death_stationary, birth_death_generator, is_generator, stationary_ctmc
from spkit.plots import plot_hist_vs_pdf

SEED, rng = rng_for("03e")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기 (이 노트북에서는 손님 수의 척도로만 참고)
N_CUST = scale(100_000, divisor=5)   # 손님 단위 시뮬레이션의 손님 수 (FAST: 20_000 — rho=0.75 자기상관 때문에 1/10보다 크게)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_CUST={N_CUST}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (03d) $\rho=\lambda/\mu$인 M/M/1의 $L, W, L_q, W_q$는?

<details><summary>정답</summary>

$L=\frac{\rho}{1-\rho}$, $W=\frac1{\mu-\lambda}$, $L_q=\frac{\rho^2}{1-\rho}$, $W_q=\frac{\rho}{\mu-\lambda}$. ($W=W_q+1/\mu$, $L=\lambda W$로 서로 이어진다.)

</details>

**Q2.** (03d) 리틀의 법칙과 그것이 요구하는 가정은?

<details><summary>정답</summary>

$L=\lambda W$. 분포·서비스 규칙·서버 수와 무관한 회계 항등식이고, 극한의 존재(안정성)만 필요하다. 표본경로 버전은 $\int_0^TN(t)\,dt=\sum_iW_i$ — 계단함수 아래 면적을 세로로 세나 가로로 세나 같다.

</details>

**Q3.** (03d) PASTA는 무엇이고 어떤 도착에서 성립하는가?

<details><summary>정답</summary>

푸아송 도착이 보는 상태의 장기 경험분포 = 시간평균 분포 $\pi$. 푸아송(미래를 예견하지 않는 독립 증분) 도착에서만 성립한다; D/M/1에서는 도착이 보는 빈 시스템 비율이 $\pi_0$보다 크다.

</details>

**Q4.** (03c) M/M/∞의 정상분포는 무엇이고 곱 형태의 어느 항에서 나오나?

<details><summary>정답</summary>

$\mathrm{Poisson}(\lambda/\mu)$. 곱 형태 $\pi_n\propto\prod_{k<n}\frac{\lambda_k}{\mu_{k+1}}$에서 $\mu_{k+1}=(k+1)\mu$이므로 $\prod_{k<n}\frac{\lambda}{(k+1)\mu}=\frac{(\lambda/\mu)^n}{n!}$ — 푸아송 pmf의 모양이 그대로 나온다. 이 노트북의 $n\le c$ 구간은 정확히 이 항이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **M/M/c**: 비율 $\lambda$의 푸아송 도착, 서버 $c$대가 각각 i.i.d. $\mathrm{Exp}(\mu)$ 서비스, **공용** FIFO 대기열(빈 서버가 생기면 줄의 맨 앞 손님이 들어간다). 시스템 내 손님 수 $N(t)$는 $\lambda_n=\lambda$, $\mu_n=\min(n,c)\mu$인 출생-사망 CTMC다. **제공 부하(offered load)** $a=\lambda/\mu$ (단위 Erlang: '평균적으로 몇 대의 서버가 필요한가'), **서버당 이용률** $\rho=a/c=\lambda/(c\mu)$. 안정성 조건은 $\rho<1$, 즉 $a<c$.
- **M/M/c 정상분포**: $\pi_n=\pi_0\frac{a^n}{n!}$ ($n\le c$), $\pi_n=\pi_0\frac{a^c}{c!}\rho^{n-c}$ ($n\ge c$), $\pi_0^{-1}=\sum_{n=0}^{c-1}\frac{a^n}{n!}+\frac{a^c}{c!(1-\rho)}$. $n\le c$에서는 M/M/∞(푸아송 모양), $n\ge c$에서는 M/M/1(기하 꼬리) — 두 노트북의 답을 $n=c$에서 이어 붙인 것이다.
- **Erlang C**: $C(c,a)=P_\pi(N\ge c)=\pi_0\frac{a^c}{c!(1-\rho)}$. PASTA에 의해 = 도착 손님이 **기다릴** 확률. 그로부터 $L_q=C\frac{\rho}{1-\rho}$, $W_q=L_q/\lambda$, $W=W_q+1/\mu$, $L=L_q+a$. spkit의 `mmc_metrics(lam, mu, c)`가 keys `rho, p_wait, Lq, L, Wq, W`로 이 값들을 준다.
- **대기 조건부 대기시간**: $W_q\mid W_q>0\sim\mathrm{Exp}(c\mu-\lambda)$, 따라서 $P(W_q>t)=C\,e^{-(c\mu-\lambda)t}$ (03d 정리 4의 $c$대 버전).
- **M/M/c/K**: 시스템에 최대 $K\ge c$명(서비스 중 $c$ + 대기 $K-c$). $\lambda_n=\lambda$ ($n<K$), $\lambda_K=0$, $\mu_n=\min(n,c)\mu$. 자리가 없을 때 도착한 손님은 **차단**(blocked)되어 사라진다. 차단 확률 $P_B=\pi_K$ (PASTA), **유효 도착률** $\lambda_{\rm eff}=\lambda(1-\pi_K)$, 리틀은 $L=\lambda_{\rm eff}W$ (들어온 손님만 센다). 상태가 유한하므로 안정성 조건이 없다.
- **Erlang B** (손실 시스템 M/M/c/c, $K=c$, 대기 없음): $B(c,a)=\dfrac{a^c/c!}{\sum_{n=0}^ca^n/n!}$. 재귀 $B(0,a)=1$, $B(k,a)=\dfrac{a\,B(k-1,a)}{k+a\,B(k-1,a)}$는 큰 $c$에서 $c!$ 오버플로를 피한다.
- **이 노트북의 수치**: (i) M/M/3, $\lambda=2,\mu=1$ ($a=2$, $\rho=2/3$): $\pi_0=1/9$, $C=4/9$, $L_q=8/9$, $W_q=4/9$, $W=13/9$, $L=26/9$. (ii) M/M/3/3, $a=2$: $B=4/19$. (iii) M/M/2/5, $\lambda=2,\mu=1$: $\pi=(1,2,2,2,2,2)/11$, $\pi_5=2/11$, $L=30/11$, $\lambda_{\rm eff}=18/11$, $W=5/3$. (iv) $\lambda=1.5$: 빠른 1대($\mu=2$) $W=2$, $W_q=1.5$ vs 느린 2대($\mu=1$) $\pi_0=1/7$, $C=9/14$, $L_q=27/14$, $W_q=9/7$, $W=16/7$.

### 2.2 정리 1 — M/M/c 정상분포와 Erlang C (Durrett 3e §4.5, Ross 12e §8.9.2, Norris §5.2.2)

> $\rho=a/c<1$이면 위 정의의 $\pi_n$이 정상분포이고, $C(c,a)=\pi_0\frac{a^c}{c!(1-\rho)}$, $L_q=C\frac{\rho}{1-\rho}$, $L=L_q+a$.

**가정이 왜 필요한가**
- **$\rho<1$**: $n\ge c$에서 $\pi_n\propto\rho^{n-c}$의 합 $\sum\rho^{n-c}$이 유한해야 정규화된다. 조건은 $a<c$이지 $a<1$이 아니다 — $a=2$도 서버가 3대면 안정이다.
- **공용 대기열 + 가장 먼저 비는 서버에 배정**: 서버별로 줄이 따로 있으면(마트 계산대) 상태가 '각 줄의 길이' 벡터가 되어 출생-사망이 아니다. 이 가정이 곧 "빈 서버가 있는데 기다리는 사람은 없다"는 뜻이고, 그래서 $\mu_n=\min(n,c)\mu$가 된다.
- **지수 서비스**: 서비스 중인 $\min(n,c)$명 중 누가 먼저 끝나든 **총 완료율**이 $\min(n,c)\mu$인 것은 독립 지수 시계의 최소가 $\mathrm{Exp}(\sum\text{rates})$라는 03a 정리 1(경쟁하는 시계) 덕분이다.

<details><summary>증명</summary>

03c 정리 2의 곱 형태 $\pi_n=\pi_0\prod_{k<n}\frac{\lambda_k}{\mu_{k+1}}$에 $\lambda_k=\lambda$, $\mu_{k+1}=\min(k+1,c)\mu$를 넣는다. $n\le c$이면 $\prod_{k<n}\frac{\lambda}{(k+1)\mu}=\frac{a^n}{n!}$; $n>c$이면 처음 $c$개 인자가 $\frac{a^c}{c!}$를 주고 나머지 $n-c$개 인자는 각각 $\frac{\lambda}{c\mu}=\rho$이므로 $\frac{a^c}{c!}\rho^{n-c}$. 정규화: $\sum_{n<c}\frac{a^n}{n!}+\frac{a^c}{c!}\sum_{n\ge c}\rho^{n-c}=\sum_{n<c}\frac{a^n}{n!}+\frac{a^c}{c!(1-\rho)}=\pi_0^{-1}$ ($\rho<1$).

**Erlang C**: $C=\sum_{n\ge c}\pi_n=\pi_0\frac{a^c}{c!}\sum_{m\ge0}\rho^m=\pi_0\frac{a^c}{c!(1-\rho)}$. PASTA(03d 정리 3)로 이것이 도착 손님이 '모든 서버가 바쁜 상태'를 볼 확률, 즉 기다릴 확률이다.

**$L_q$**: 대기 중인 손님은 $(N-c)^+$명이므로
$$L_q=\sum_{n\ge c}(n-c)\pi_n=\pi_0\frac{a^c}{c!}\sum_{m\ge0}m\rho^m=\pi_0\frac{a^c}{c!}\cdot\frac{\rho}{(1-\rho)^2}=C\cdot\frac{\rho}{1-\rho}.$$
(03d와 같은 기하급수 미분 $\sum m\rho^m=\rho/(1-\rho)^2$.)

**$L$**: 서비스 중인 평균 손님 수는 리틀의 법칙을 '서버들'이라는 하위 시스템에 적용해 $\lambda\cdot E[\text{서비스}]=\lambda/\mu=a$. 따라서 $L=L_q+a$. (직접 계산 $\sum_n\min(n,c)\pi_n=a$로도 같은 결론 — 절단 방정식 $\lambda\pi_n=\min(n+1,c)\mu\,\pi_{n+1}$을 $n$에 대해 더하면 $\lambda=\mu\sum_n\min(n,c)\pi_n$.) $W_q=L_q/\lambda$, $W=W_q+1/\mu$는 리틀이다.

수치 (i) $c=3,a=2$: $\sum_{n<3}a^n/n!=1+2+2=5$, $\frac{a^3}{3!(1-\rho)}=\frac{8}{6\cdot\frac13}=4$, $\pi_0^{-1}=9$, $C=4\pi_0=4/9$, $L_q=\frac49\cdot\frac{2/3}{1/3}=\frac89$, $W_q=\frac49$, $W=\frac{13}9$, $L=\frac89+2=\frac{26}9$. $\square$

</details>

### 2.3 정리 2 — M/M/c/K와 Erlang B; 무감각성 (Durrett 3e §4.5, Ross 12e §8.3·§8.9.1, Norris §5.2.3)

> M/M/c/K의 정상분포는 같은 곱 형태를 $n\le K$로 절단·재정규화한 것이고 $P_B=\pi_K$. $K=c$이면 $\pi_n=\dfrac{a^n/n!}{\sum_{k\le c}a^k/k!}$ (절단 푸아송)이고 $P_B=B(c,a)$. 나아가 Erlang B는 서비스 분포에 **무감각**(insensitive)하다: 서비스가 평균 $E[S]$인 임의 분포인 M/G/c/c도 같은 $B(c,a)$, $a=\lambda E[S]$.

**가정이 왜 필요한가**
- **유한 $K$**: 합이 유한하므로 $\rho<1$ 조건이 필요 없다 — 과부하($a\ge c$) 손실 시스템도 정상분포를 가진다. 대신 $\lambda\ne\lambda_{\rm eff}$라 리틀을 쓸 때 조심해야 한다.
- **차단된 손님은 사라짐(재시도 없음)**: 차단된 손님이 잠시 뒤 다시 오면 도착 과정이 더 이상 푸아송이 아니고 PASTA($P_B=\pi_K$)가 깨진다.
- **무감각성은 손실 시스템($K=c$)에서만**: M/M/c(대기 있음)의 Erlang C는 서비스 분포에 민감하다 — E3에서 결정론적 서비스로 바꾸면 $W_q$가 약 절반이 된다.

<details><summary>증명</summary>

**절단.** M/M/c/K는 상태 $\{0,\dots,K\}$의 유한 출생-사망 사슬이고 $\lambda_K=0$이다. 03c 정리 2의 곱 형태는 상태 공간이 유한해도 그대로 성립하므로($\pi_n=\pi_0\prod_{k<n}\lambda_k/\mu_{k+1}$, $n\le K$), M/M/c의 미정규화 항 $\frac{a^n}{n!}$ ($n\le c$), $\frac{a^c}{c!}\rho^{n-c}$ ($c\le n\le K$)를 $n\le K$까지만 더해 정규화하면 된다. 차단은 '도착이 상태 $K$를 보는' 사건이므로 PASTA로 $P_B=\pi_K$.

**$K=c$.** $\mu_n=n\mu$뿐이므로 $\pi_n\propto a^n/n!$ — M/M/∞의 푸아송을 $n\le c$에서 절단한 것. $P_B=\pi_c=\frac{a^c/c!}{\sum_{k\le c}a^k/k!}=B(c,a)$. 재귀식: $S_{k-1}=\sum_{j<k}a^j/j!$로 두면 $B(k-1,a)=\frac{a^{k-1}/(k-1)!}{S_{k-1}}$이고 $B(k,a)=\frac{a^k/k!}{S_{k-1}+a^k/k!}=\frac{x_k}{1+x_k}$, $x_k:=\frac{a^k/k!}{S_{k-1}}=\frac ak\cdot\frac{a^{k-1}/(k-1)!}{S_{k-1}}=\frac ak\,B(k-1,a)$. 대입하면 $B(k,a)=\frac{aB(k-1,a)/k}{1+aB(k-1,a)/k}=\frac{aB(k-1,a)}{k+aB(k-1,a)}$.

**무감각성**은 인용한다(Ross 12e §8.9.1; Sevastyanov 1957): M/G/c/c에서 '서비스 중인 손님 수'의 정상분포는 서비스 분포의 평균에만 의존한다. 직관은 M/G/∞에서 시각 $t$의 손님 수가 $\mathrm{Poisson}(\lambda E[S])$인 것(02c의 푸아송 세분화, 평균만 남는다)과 같은 이유이고, $K=c$ 절단은 그 푸아송을 자르는 것이다. 이 노트북에서는 E3에서 결정론적 서비스로 수치 확인만 한다.

수치: $B(3,2)=\frac{8/6}{1+2+2+8/6}=\frac{4/3}{19/3}=\frac4{19}$. M/M/2/5 ($a=2$): 비 $\lambda/\mu_n=\frac21,\frac22,\frac22,\frac22,\frac22$ → $\pi\propto(1,2,2,2,2,2)$, $\pi_5=\frac2{11}$, $L=\frac{0+2+4+6+8+10}{11}=\frac{30}{11}$, $\lambda_{\rm eff}=2\cdot\frac9{11}=\frac{18}{11}$, $W=L/\lambda_{\rm eff}=\frac53$. $\square$

</details>

### 2.4 정리 3 — 대기 조건부 대기시간 (Ross 12e §8.9.2)

> 정상 상태 FIFO M/M/c에서 $P(W_q>t)=C(c,a)\,e^{-(c\mu-\lambda)t}$; 즉 $W_q\mid W_q>0\sim\mathrm{Exp}(c\mu-\lambda)$이고, 평균 $E[W_q]=C/(c\mu-\lambda)$는 $L_q/\lambda$와 일치한다.

**가정이 왜 필요한가**
- **PASTA**: 도착이 보는 $N$의 분포가 $\pi$여야 한다. 특히 $N\ge c$일 때 $N-c\sim\mathrm{Geom}(\rho)$ ($P(N-c=m\mid N\ge c)=(1-\rho)\rho^m$).
- **지수 서비스 + $c$대 모두 바쁨**: 다음 완료까지의 시간이 $\mathrm{Exp}(c\mu)$ (경쟁하는 시계)이고, 무기억성 덕분에 완료가 일어날 때마다 새로 $\mathrm{Exp}(c\mu)$에서 시작한다.
- **FIFO**: 내 앞의 $N-c$명이 모두 서비스에 들어간 뒤 한 번 더 완료가 일어나야 내 차례다 — 완료 $N-c+1$번을 기다린다.

<details><summary>증명</summary>

PASTA로 도착 손님이 $N=n\ge c$를 볼 확률은 $\pi_n=C(1-\rho)\rho^{n-c}$ (정리 1의 $\pi_n=\pi_0\frac{a^c}{c!}\rho^{n-c}$와 $C=\pi_0\frac{a^c}{c!(1-\rho)}$). 이때 서버 $c$대가 모두 바쁘고, 손님은 완료 $m+1$번($m=n-c$)을 기다린다. 대기 중에는 항상 $c$대가 바쁘므로 완료 간격은 i.i.d. $\mathrm{Exp}(c\mu)$, 조건부 대기시간은 $\mathrm{Erlang}(m+1,c\mu)$. 03d 정리 4와 같은 라플라스 변환 계산:
$$E\big[e^{-sW_q};W_q>0\big]=\sum_{m\ge0}C(1-\rho)\rho^m\Big(\frac{c\mu}{c\mu+s}\Big)^{m+1}=C\,\frac{(1-\rho)c\mu}{c\mu+s}\cdot\frac1{1-\rho\,c\mu/(c\mu+s)}=C\,\frac{c\mu(1-\rho)}{c\mu(1-\rho)+s},$$
그리고 $c\mu(1-\rho)=c\mu-\lambda$. 이는 $C\times[\mathrm{Exp}(c\mu-\lambda)$의 변환$]$이므로 $W_q\mid W_q>0\sim\mathrm{Exp}(c\mu-\lambda)$, $P(W_q>0)=C$, $P(W_q>t)=Ce^{-(c\mu-\lambda)t}$.

평균: $E[W_q]=\int_0^\infty P(W_q>t)\,dt=\frac{C}{c\mu-\lambda}=\frac{C}{c\mu(1-\rho)}=\frac{C\rho}{(1-\rho)\lambda}=\frac{L_q}{\lambda}$ ✓ (리틀과 일치). 수치 (i): $c\mu-\lambda=1$, 조건부 평균 대기 $=1$, $E[W_q]=4/9$. $\square$

</details>

### 2.5 정리 4 — 빠른 1대 vs 느린 2대 (Ross 12e §8.9.2 참고)

> 같은 $\lambda$에서 M/M/1(서비스율 $2\mu$)과 M/M/2(각 $\mu$)를 비교하면, 모든 $0<\lambda<2\mu$에 대해 $W_{\rm fast}<W_{\rm two}$이지만 $W_{q,\rm two}<W_{q,\rm fast}$이다. 구체적으로 $\rho=\lambda/(2\mu)$일 때 $\dfrac{W_{\rm two}}{W_{\rm fast}}=\dfrac2{1+\rho}>1$, $\dfrac{W_{q,\rm two}}{W_{q,\rm fast}}=\dfrac{2\rho}{1+\rho}<1$.

**가정이 왜 필요한가**
- **같은 총 용량 $2\mu$**: 비교가 공정하려면 두 설계의 처리 능력(따라서 $\rho$)이 같아야 한다.
- **$W_q$는 '기다리는 시간'만, $W$는 서비스 포함**: 느린 서버의 서비스 $1/\mu$가 빠른 서버의 $1/(2\mu)$보다 $1/(2\mu)$만큼 길고, 이 차이가 대기시간의 이득을 뒤집는다.

<details><summary>증명</summary>

$\rho=\lambda/(2\mu)<1$로 두자.

**빠른 1대** (M/M/1, 비율 $2\mu$, 이용률 $\rho$): $W_{\rm fast}=\frac1{2\mu-\lambda}=\frac{1}{2\mu(1-\rho)}$, $W_{q,\rm fast}=W_{\rm fast}-\frac1{2\mu}=\frac{\rho}{2\mu(1-\rho)}$.

**느린 2대** (M/M/2, $a=\lambda/\mu=2\rho$, 서버당 이용률 $a/2=\rho$): 정리 1에서
$\pi_0^{-1}=1+2\rho+\frac{(2\rho)^2}{2(1-\rho)}=\frac{(1+2\rho)(1-\rho)+2\rho^2}{1-\rho}=\frac{1+\rho}{1-\rho}$,
$C=\pi_0\frac{(2\rho)^2}{2(1-\rho)}=\frac{1-\rho}{1+\rho}\cdot\frac{2\rho^2}{1-\rho}=\frac{2\rho^2}{1+\rho}$,
$L_q=C\frac{\rho}{1-\rho}=\frac{2\rho^3}{1-\rho^2}$,
$W_{q,\rm two}=\frac{L_q}{\lambda}=\frac{2\rho^3}{2\mu\rho(1-\rho^2)}=\frac{\rho^2}{\mu(1-\rho^2)}$,
$W_{\rm two}=W_{q,\rm two}+\frac1\mu=\frac{\rho^2+(1-\rho^2)}{\mu(1-\rho^2)}=\frac1{\mu(1-\rho^2)}$.

**비율**: $\frac{W_{\rm two}}{W_{\rm fast}}=\frac{2\mu(1-\rho)}{\mu(1-\rho)(1+\rho)}=\frac2{1+\rho}>1$, $\frac{W_{q,\rm two}}{W_{q,\rm fast}}=\frac{\rho^2}{\mu(1-\rho^2)}\cdot\frac{2\mu(1-\rho)}{\rho}=\frac{2\rho}{1+\rho}<1$ — 모든 $0<\rho<1$에서. $\rho\to1$이면 두 비율 모두 1로 수렴(과부하에서는 어느 설계든 대기가 지배), $\rho\to0$이면 $W$ 비율은 2(빈 시스템에서는 서비스 시간만 남는다), $W_q$ 비율은 0(느린 2대는 거의 기다리지 않는다).

수치 (iv) $\lambda=1.5,\mu=1$ ($\rho=0.75$): 빠른 1대 $W=\frac1{2-1.5}=2$, $W_q=2-\frac12=1.5$; 느린 2대 $\pi_0=\big[1+1.5+\frac{2.25}{2\cdot0.25}\big]^{-1}=\frac17$, $C=\frac{4.5}{7}=\frac9{14}$, $L_q=\frac9{14}\cdot\frac{0.75}{0.25}=\frac{27}{14}$, $W_q=\frac{27/14}{1.5}=\frac97$, $W=\frac{16}7$. 비율 $\frac{16/7}{2}=\frac87=\frac2{1.75}$ ✓, $\frac{9/7}{1.5}=\frac67=\frac{1.5}{1.75}$ ✓. $\square$

</details>

### 2.6 직관

**풀링(pooling)의 두 얼굴.** 느린 서버 2대는 손님이 1명일 때 한 대가 놀지만, 그 대신 새 손님이 '놀고 있는 서버'로 바로 들어가므로 기다리는 시간 $W_q$는 짧다. 빠른 서버 1대는 일단 잡히면 두 배 빨리 끝내므로 서비스 포함 체류 $W$는 짧다. 설계 질문이 '대기'인지 '체류'인지에 따라 답이 다르다 — 콜센터(응답 대기 시간이 지표)와 응급실(총 체류 시간이 지표)이 다른 설계를 고르는 이유다.

**Erlang C vs B.** C는 '도착해 보니 서버가 모두 바쁠 확률', B는 '도착해 보니 자리가 없어 쫓겨날 확률'. B는 서비스 분포의 **평균만** 보고(무감각성), C는 분산에도 민감하다 — 손실 시스템에는 '기다리며 분산이 누적되는' 대기열이 없기 때문이다. 대기열은 서비스 시간의 변동을 뒤 손님에게 전달하는 통로이고, 그 통로를 막으면(K=c) 분포의 모양이 사라지고 평균만 남는다.

**서버 수의 힘.** 콜센터 설계(C2)의 핵심은 $C(c,a)$가 $c$에 대해 거의 지수적으로 떨어진다는 것이다: $a=2$일 때 $c=3\to8$에서 $0.44\to0.001$. 서버를 하나 더 두는 것이 서버를 조금 빠르게 하는 것보다 대기 확률을 훨씬 크게 줄인다.

### 2.7 함정(traps)

1. **Erlang C의 $\rho$는 $a/c=\lambda/(c\mu)$**이지 $\lambda/\mu$가 아니다. 안정성은 $a<c$; $a=2$는 $c\ge3$이면 안정이다.
2. **$L=L_q+a$** (서비스 중 평균 손님 수 $=\lambda/\mu$, 리틀); $L_q+c$가 아니다 — 서버가 모두 바쁜 것이 아니다.
3. **유한 용량에서 리틀은 $\lambda_{\rm eff}=\lambda(1-\pi_K)$로**: $W=L/\lambda_{\rm eff}$. $L/\lambda$를 쓰면 체류시간을 과소평가한다(M/M/2/5: $30/22\approx1.36$ vs 정답 $5/3$).
4. **Erlang B는 서비스 분포에 무감각하지만 Erlang C(와 $W_q$)는 아니다.** M/G/c는 닫힌 형식이 없어 근사·시뮬레이션(C2)으로 다룬다.
5. **차단 비율(손님 관점) = $\pi_K$(시간 관점)는 PASTA 덕분.** 도착이 푸아송이 아니면 두 값이 다르다.
6. **다중 서버 FIFO 시뮬레이션은 '가장 먼저 비는 서버'(힙의 최소)에 배정**해야 M/M/c다. 임의 서버 배정이나 서버별 개별 줄은 다른(더 나쁜) 시스템이다.
7. **`factorial`은 큰 $c$에서 오버플로**(부동소수점으로 $c\gtrsim170$) — Erlang B는 재귀식으로, Erlang C는 `mmc_metrics`처럼 항을 누적 곱으로 계산한다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요. (a) $L=L_q+a$에서 $a$가 왜 $c$나 $\rho$가 아니라 $\lambda/\mu$인지 리틀의 법칙으로 설명하라. (b) '빠른 1대가 항상 낫다'는 주장이 틀린 지표를 하나 들어라.)

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향(특히 마지막 두 항목의 대소)이 맞는지가 목적입니다.

In [ ]:
predictions = {
    # M/M/3, lambda=2, mu=1 에서 도착 손님이 기다릴 확률 (Erlang C) 은?
    "p_wait_c3": None,
    # 같은 시스템의 평균 대기시간 W_q 는?
    "Wq_c3": None,
    # 전화 회선 3개, 부하 a=2 Erlang (M/M/3/3) 에서 통화가 차단될 확률은?
    "erlang_b_3_2": None,
    # M/M/2/5, lambda=2, mu=1 에서 차단 확률 pi_5 는?
    "block_mm2_5": None,
    # lambda=1.5, 서버 1대 mu=2: 평균 체류시간 W 는?
    "W_one_fast": None,
    # lambda=1.5, 서버 2대 각 mu=1: 평균 체류시간 W 는? (빠른 1대보다 큰가 작은가?)
    "W_two_slow": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 도구: 다중 서버 손님 시뮬레이션과 Erlang 공식

03d의 Lindley 재귀를 $c$대로 확장한다. 각 서버가 **비는 시각**을 최소 힙 `free`에 두면, 힙의 최소가 '가장 먼저 비는 서버'이고 손님 $i$의 서비스 시작은 $\max(A_i,\min\text{free})$ — 이것이 공용 FIFO 대기열이다(트랩 6). 유한 용량 $K$는 시스템 내 손님들의 퇴장 시각을 담은 두 번째 힙으로 검사한다(도착 시각 이전에 떠난 손님을 먼저 빼낸다). 차단된 손님은 `start = D = nan`.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def mmck_customers(lam, mu, c, n_customers, rng, K=None, service=None):
    """FIFO M/M/c (K=None) 또는 M/M/c/K 손님 단위 시뮬레이션. 가장 먼저 비는 서버에 배정 (힙).
    returns (A, start, D); 차단된 손님은 start = D = nan.
    service: None이면 Exp(1/mu); 아니면 service(n, rng) -> 서비스시간 배열 (E3용)."""
    A = np.cumsum(rng.exponential(1.0 / lam, n_customers))
    S = rng.exponential(1.0 / mu, n_customers) if service is None else service(n_customers, rng)
    D = np.full(n_customers, np.nan); start = np.full(n_customers, np.nan)
    free = [0.0] * c          # 각 서버가 비는 시각 (min-heap)
    in_system = []            # 시스템 내 손님의 퇴장 시각 (min-heap), K 검사용
    for i in range(n_customers):
        t = A[i]
        if K is not None:
            while in_system and in_system[0] <= t:
                heapq.heappop(in_system)
            if len(in_system) >= K:
                continue      # 차단
        f = heapq.heappop(free); s = max(t, f); d = s + S[i]
        heapq.heappush(free, d); start[i] = s; D[i] = d
        if K is not None:
            heapq.heappush(in_system, d)
    return A, start, D

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def batch_means_se(x, n_batches=20):
    """자기상관 있는 수열의 평균에 대한 SE (배치 평균). 03d와 동일."""
    x = np.asarray(x, float); m = len(x) // n_batches
    b = x[: m * n_batches].reshape(n_batches, m).mean(axis=1)
    return b.std(ddof=1) / np.sqrt(n_batches)

def number_in_system(A, D):
    """도착·퇴장 시각으로 N(t) 계단함수 (t_events, N) 생성 (차단된 손님 = nan 제외)."""
    ok = ~np.isnan(D)
    t = np.concatenate([A[ok], D[ok]]); d = np.concatenate([np.ones(ok.sum()), -np.ones(ok.sum())])
    o = np.argsort(t, kind="stable")
    return t[o], np.cumsum(d[o])

def erlang_b(c, a):
    """Erlang B: M/M/c/c 차단 확률, a = lam/mu (Erlang 단위 부하)."""
    return (a**c / factorial(c)) / sum(a**n / factorial(n) for n in range(c + 1))

def erlang_b_recursive(c, a):
    """같은 값을 재귀식으로 (큰 c에서 factorial 오버플로 방지, 트랩 7)."""
    B = 1.0
    for k in range(1, c + 1):
        B = a * B / (k + a * B)
    return B

def erlang_c(c, a):
    """Erlang C: M/M/c 대기 확률 P(N >= c), a < c 필요."""
    rho = a / c
    tail = a**c / (factorial(c) * (1 - rho))
    return tail / (sum(a**n / factorial(n) for n in range(c)) + tail)

### 4.2 표본경로: M/M/3의 $N(t)$ (단계 2a)

$\lambda=2,\mu=1,c=3$으로 `N_CUST`명을 시뮬레이션하고 처음 60 시간 단위의 $N(t)$를 그린다. $N(t)\le3$이면 아무도 기다리지 않고, $N(t)>3$인 구간의 높이 $N-3$이 대기열 길이다. 점선은 $L=26/9$와 서버 수 $c=3$.

In [ ]:
LAM, MU, C = 2.0, 1.0, 3
A, start, D = mmck_customers(LAM, MU, C, N_CUST, rng)
Wq, W = start - A, D - A                      # 대기시간, 체류시간 (손님별)
warm = N_CUST // 10
t_ev, N = number_in_system(A, D)
m3 = mmc_metrics(LAM, MU, C)

T_SHOW = 60.0
k = np.searchsorted(t_ev, T_SHOW)
fig, ax = plt.subplots()
ax.step(t_ev[:k], N[:k], where="post", lw=1)
ax.fill_between(t_ev[:k], np.minimum(N[:k], C), step="post", alpha=0.25, label="in service (<= c)")
ax.fill_between(t_ev[:k], C, np.maximum(N[:k], C), step="post", alpha=0.35, color="C3", label="waiting (N - c)")
ax.axhline(C, color="k", ls=":", lw=1, label="c = 3 servers")
ax.axhline(m3["L"], color="C3", ls="--", lw=1, label=f"L = 26/9 = {m3['L']:.3f}")
ax.set_xlabel("t"); ax.set_ylabel("N(t): customers in system")
ax.set_title("Fig 1: M/M/3 (lambda = 2, mu = 1): number in system, first 60 time units")
ax.legend(loc="upper right")
plt.show()
frac_busy = np.sum(np.diff(t_ev) * (N[:-1] >= C)) / (t_ev[-1] - t_ev[0])   # 시간 평균 (측정)
print(f"n = {N_CUST} customers, T = {D[-1]:.1f}, max N(t) = {int(N.max())}")
print(f"fraction of time N >= {C}: measured {frac_busy:.4f}  vs  Erlang C (closed form) {m3['p_wait']:.4f}")

빨간 면적이 '기다리는 손님 수'의 시간 적분이고, $[0,T]$ 전체에서 $T$로 나누면 $L_q=8/9$ (리틀: $\lambda W_q$)에 가까워야 한다. 비어 있는 서버가 있는데 기다리는 사람이 있는 구간은 없다 — 힙 배정이 이를 보장한다.

### 4.3 닫힌 형식과 spkit 대조 (단계 1)

정리 1·2의 값이 `mmc_metrics`, 인라인 Erlang 공식, 그리고 출생-사망 곱 형태 `birth_death_stationary`와 부동소수점 수준에서 일치해야 한다. 여기서 확인되는 것은 **대수**(같은 공식의 세 가지 구현)이지 확률적 검증이 아니다.

In [ ]:
# M/M/3, a = 2: 정리 1
print("mmc_metrics(2, 1, 3) =", {k: round(v, 6) for k, v in m3.items()})
np.testing.assert_allclose([m3["p_wait"], m3["Lq"], m3["Wq"], m3["W"], m3["L"]],
                           [4 / 9, 8 / 9, 4 / 9, 13 / 9, 26 / 9], rtol=1e-10)
np.testing.assert_allclose(erlang_c(3, 2.0), m3["p_wait"], rtol=1e-12)

# M/M/3/3: Erlang B 두 구현 + 출생-사망 곱 형태 (birth[n] = lam, death[n] = n mu, n = 0..3)
pi_333 = birth_death_stationary(np.full(3, LAM), np.arange(4) * MU)
print(f"Erlang B(3, 2): closed {erlang_b(3, 2.0):.6f}, recursive {erlang_b_recursive(3, 2.0):.6f}, "
      f"birth-death pi_3 {pi_333[3]:.6f}   (4/19 = {4/19:.6f})")
np.testing.assert_allclose([erlang_b(3, 2.0), erlang_b_recursive(3, 2.0), pi_333[3]], 4 / 19, rtol=1e-10)

# M/M/2/5: 절단 곱 형태, 생성원으로도 확인 (death[n] = min(n, 2) mu)
pi_mm25 = birth_death_stationary(np.full(5, LAM), np.minimum(np.arange(6), 2) * MU)
Q25 = birth_death_generator(np.full(5, LAM), np.minimum(np.arange(6), 2) * MU)
assert is_generator(Q25)
np.testing.assert_allclose(pi_mm25, np.array([1, 2, 2, 2, 2, 2]) / 11, atol=1e-12)
np.testing.assert_allclose(stationary_ctmc(Q25), pi_mm25, atol=1e-12)
L_25 = np.sum(np.arange(6) * pi_mm25); lam_eff_25 = LAM * (1 - pi_mm25[5]); W_25_exact = L_25 / lam_eff_25
print(f"M/M/2/5: pi = {np.round(pi_mm25, 4)}, pi_5 = {pi_mm25[5]:.6f} (2/11), L = {L_25:.6f} (30/11), "
      f"lam_eff = {lam_eff_25:.6f} (18/11), W = L/lam_eff = {W_25_exact:.6f} (5/3)")

$K\to\infty$ 극한: M/M/3/K의 $\pi_K$는 0으로 가고, 절단분포의 $\sum_{n\ge3}\pi_n$은 Erlang C $=4/9$로 수렴한다 — 유한 용량 시스템은 대기실을 키우면 M/M/c가 된다.

In [ ]:
print("| K | pi_K (block) | sum_{n>=3} pi_n | Erlang C(3,2) |\n|---|---|---|---|")
for K in (3, 4, 5, 10, 20, 40):
    piK = birth_death_stationary(np.full(K, LAM), np.minimum(np.arange(K + 1), C) * MU)
    print(f"| {K} | {piK[K]:.2e} | {piK[C:].sum():.6f} | {m3['p_wait']:.6f} |")

서버 수 $c$를 늘리면 $C(c,2)$와 $B(c,2)$가 얼마나 빨리 떨어지는가 — 세미로그에서 거의 직선이다(2.6 직관).

In [ ]:
cs = np.arange(3, 9)
C_vals = np.array([mmc_metrics(LAM, MU, c)["p_wait"] for c in cs])
B_vals = np.array([erlang_b_recursive(c, LAM / MU) for c in cs])
fig, ax = plt.subplots()
ax.semilogy(cs, C_vals, "o-", label="Erlang C(c, 2): P(arrival waits), M/M/c")
ax.semilogy(cs, B_vals, "s--", label="Erlang B(c, 2): P(arrival blocked), M/M/c/c")
for c, cv, bv in zip(cs, C_vals, B_vals):
    ax.annotate(f"{cv:.3g}", (c, cv), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("number of servers c"); ax.set_ylabel("probability (log scale)")
ax.set_title("Fig 2: Erlang C (wait) and Erlang B (block) vs c, offered load a = 2")
ax.legend()
plt.show()
print("C(c,2) ratio between consecutive c:", np.round(C_vals[1:] / C_vals[:-1], 3))

### 4.4 M/M/3 측정: 대기 확률, $W_q$, $W$, 조건부 대기 (단계 2b)

4.2에서 만든 손님 자료의 처음 10%를 워밍업으로 버리고 나머지로 잰다. SE는 모두 **배치 평균**(20개 블록; 03d 트랩 2). 조건부 대기 $W_q\mid W_q>0$의 평균은 정리 3에 따라 $1/(c\mu-\lambda)=1$이어야 한다.

In [ ]:
Wqw, Ww = Wq[warm:], W[warm:]
waited = (Wqw > 0).astype(float)
p_wait_hat, se_pwait = waited.mean(), batch_means_se(waited, 20)
Wq_hat, se_Wq = Wqw.mean(), batch_means_se(Wqw, 20)
W_hat, se_W = Ww.mean(), batch_means_se(Ww, 20)
cond = Wqw[Wqw > 0]
cond_wait_hat, se_cond = cond.mean(), batch_means_se(cond, 20)
cond_exact = 1.0 / (C * MU - LAM)                       # 정리 3
print(f"after warm-up of {warm} customers ({N_CUST - warm} used, {len(cond)} waited):")
print(f"  P(wait)      = {p_wait_hat:.4f} ± {se_pwait:.4f}   (Erlang C = {m3['p_wait']:.4f})")
print(f"  Wq bar       = {Wq_hat:.4f} ± {se_Wq:.4f}   (exact {m3['Wq']:.4f})")
print(f"  W bar        = {W_hat:.4f} ± {se_W:.4f}   (exact {m3['W']:.4f})")
print(f"  E[Wq | Wq>0] = {cond_wait_hat:.4f} ± {se_cond:.4f}   (exact 1/(c mu - lambda) = {cond_exact:.4f})")
print(f"  iid SE of Wq would be {mc_estimate(Wqw).se:.4f}: batch/iid ratio = {se_Wq / mc_estimate(Wqw).se:.1f}")

In [ ]:
x = np.linspace(0, cond.max(), 400)
rate = C * MU - LAM
ax = plot_hist_vs_pdf(cond, x, rate * np.exp(-rate * x), bins=60, label_pdf="Exp(c mu - lambda) pdf")
ax.set_xlabel("waiting time Wq given Wq > 0"); ax.set_ylabel("density")
ax.set_title("Fig 3: M/M/3 waiting time given wait > 0 (histogram) vs Exp(c mu - lambda), rate 1")
ax.set_xlim(0, 8)
plt.show()

히스토그램이 $e^{-x}$를 따르면 정리 3이 확인된 것이다. 03d에서는 M/M/1의 체류시간 전체가 지수였지만, M/M/c에서는 **대기시간의 양수 부분**만 지수다 — 체류시간 $W=W_q+S$는 (0에 원자를 가진 분포) + $\mathrm{Exp}(\mu)$의 합이라 지수가 아니다.

### 4.5 손실 시스템과 유한 용량 (단계 3)

M/M/3/3 ($K=c$, 대기 불가)에서 차단 비율은 Erlang B $=4/19$, M/M/2/5에서는 $\pi_5=2/11$이어야 한다(PASTA, 트랩 5). M/M/2/5에서 **들어온** 손님의 평균 체류시간은 리틀을 유효 도착률로 쓴 $L/\lambda_{\rm eff}=5/3$이다(트랩 3). 차단 비율의 SE도 배치 평균으로 — 연속한 손님의 차단 여부는 상관된다.

In [ ]:
A3, s3, D3 = mmck_customers(LAM, MU, 3, N_CUST, rng, K=3)
blocked_333 = np.isnan(D3)[warm:].astype(float)
block_333, se_b333 = blocked_333.mean(), batch_means_se(blocked_333, 20)

A5, s5, D5 = mmck_customers(LAM, MU, 2, N_CUST, rng, K=5)
blocked_25 = np.isnan(D5)[warm:].astype(float)
block_25, se_b25 = blocked_25.mean(), batch_means_se(blocked_25, 20)
W_adm = (D5 - A5)[warm:]; W_adm = W_adm[~np.isnan(W_adm)]       # 들어온 손님의 체류시간
W_25, se_W25 = W_adm.mean(), batch_means_se(W_adm, 20)
lam_eff_hat = len(W_adm) / (A5[-1] - A5[warm])                   # 유효 도착률 추정
print(f"M/M/3/3: blocked fraction = {block_333:.4f} ± {se_b333:.4f}   (Erlang B(3,2) = {4/19:.4f})")
print(f"M/M/2/5: blocked fraction = {block_25:.4f} ± {se_b25:.4f}   (pi_5 = {2/11:.4f})")
print(f"M/M/2/5: W bar of admitted = {W_25:.4f} ± {se_W25:.4f}   (L/lam_eff = {W_25_exact:.4f}; L/lam = {L_25/LAM:.4f} would be wrong)")
print(f"M/M/2/5: lam_eff hat = {lam_eff_hat:.4f} (exact {lam_eff_25:.4f}),  lam_eff hat * W bar = {lam_eff_hat * W_25:.4f} (L = {L_25:.4f})")

In [ ]:
Ks = np.arange(2, 9)
piK_c2 = np.array([birth_death_stationary(np.full(K, LAM), np.minimum(np.arange(K + 1), 2) * MU)[K] for K in Ks])
piK_c3 = np.array([birth_death_stationary(np.full(K, LAM), np.minimum(np.arange(K + 1), 3) * MU)[K] if K >= 3 else np.nan for K in Ks])
fig, ax = plt.subplots()
ax.bar(Ks - 0.2, piK_c2, width=0.4, alpha=0.7, label="closed form pi_K, c = 2")
ax.bar(Ks + 0.2, piK_c3, width=0.4, alpha=0.7, label="closed form pi_K, c = 3")
ax.errorbar([5 - 0.2], [block_25], yerr=[1.96 * se_b25], fmt="o", color="k", capsize=5, label="simulation (95% CI)")
ax.errorbar([3 + 0.2], [block_333], yerr=[1.96 * se_b333], fmt="o", color="k", capsize=5)
ax.set_xlabel("capacity K (system size)"); ax.set_ylabel("blocking probability pi_K")
ax.set_title("Fig 4: Blocking probability vs capacity K, offered load a = 2 (c = 2 and c = 3)")
ax.legend()
plt.show()

$K=c$(맨 왼쪽 막대)가 Erlang B이고, $K$를 키우면 차단이 줄지만 $c=2,a=2$($\rho=1$)에서는 $\pi\propto(1,2,2,\dots,2)$이므로 $\pi_K=2/(2K+1)\approx 1/K$로 **느리게**만 준다($K=5$: $2/11$) — 과부하 경계에서는 대기실을 늘려도 차단이 잘 사라지지 않는다. 반면 $c=3$($\rho=2/3$)에서는 기하적으로 준다.

### 4.6 빠른 1대 vs 느린 2대 (단계 4)

$\lambda=1.5$에서 M/M/1($\mu=2$)과 M/M/2($\mu=1$)를 같은 손님 수로 시뮬레이션해 $\bar W,\bar W_q$를 정리 4의 $(2,1.5)$, $(16/7,9/7)$과 비교한다. 두 시스템 모두 $\rho=0.75$라 자기상관이 강하고 SE가 크다.

In [ ]:
LAM2 = 1.5
Af, sf, Df = mmck_customers(LAM2, 2.0, 1, N_CUST, rng)           # 빠른 1대
At, st, Dt = mmck_customers(LAM2, 1.0, 2, N_CUST, rng)           # 느린 2대
Wf, Wqf = (Df - Af)[warm:], (sf - Af)[warm:]
Wt, Wqt = (Dt - At)[warm:], (st - At)[warm:]
W_fast, se_Wf = Wf.mean(), batch_means_se(Wf, 20)
Wq_fast, se_Wqf = Wqf.mean(), batch_means_se(Wqf, 20)
W_two, se_Wt = Wt.mean(), batch_means_se(Wt, 20)
Wq_two, se_Wqt = Wqt.mean(), batch_means_se(Wqt, 20)
m_fast, m_two = mmc_metrics(LAM2, 2.0, 1), mmc_metrics(LAM2, 1.0, 2)
print(f"one fast server (mu=2): W = {W_fast:.3f} ± {se_Wf:.3f} (exact {m_fast['W']:.4f}),  Wq = {Wq_fast:.3f} ± {se_Wqf:.3f} (exact {m_fast['Wq']:.4f})")
print(f"two slow servers (mu=1): W = {W_two:.3f} ± {se_Wt:.3f} (exact {m_two['W']:.4f}),  Wq = {Wq_two:.3f} ± {se_Wqt:.3f} (exact {m_two['Wq']:.4f})")
print(f"P(wait): fast {(Wqf > 0).mean():.3f} (rho = {m_fast['p_wait']:.3f}),  two {(Wqt > 0).mean():.3f} (Erlang C = {m_two['p_wait']:.4f})")

In [ ]:
fig, ax = plt.subplots()
xs = np.arange(2); wdt = 0.35
ax.bar(xs - wdt / 2, [W_fast, Wq_fast], wdt, yerr=1.96 * np.array([se_Wf, se_Wqf]), capsize=5, label="one fast server (mu = 2), simulation")
ax.bar(xs + wdt / 2, [W_two, Wq_two], wdt, yerr=1.96 * np.array([se_Wt, se_Wqt]), capsize=5, label="two slow servers (mu = 1), simulation")
ax.plot(xs - wdt / 2, [m_fast["W"], m_fast["Wq"]], "k_", ms=25, mew=2, label="closed form")
ax.plot(xs + wdt / 2, [m_two["W"], m_two["Wq"]], "k_", ms=25, mew=2)
ax.set_xticks(xs); ax.set_xticklabels(["W (sojourn)", "Wq (wait)"]); ax.set_ylabel("time")
ax.set_title("Fig 5: One fast server vs two slow servers at lambda = 1.5: W and Wq (95% CI)")
ax.legend()
plt.show()

검은 눈금(닫힌 형식)을 보면 왼쪽 묶음($W$)에서는 빠른 1대가, 오른쪽 묶음($W_q$)에서는 느린 2대가 낮다 — 순위가 뒤집힌다. 차이는 정확히 정리 4의 비율 $8/7$과 $6/7$이다. 시뮬레이션 막대는 $\rho=0.75$의 강한 자기상관 때문에 오차막대가 크고, 특히 FAST 모드(손님 2만 명)에서는 $W_q$의 차이 $1.5-9/7\approx0.21$이 SE와 비슷해 막대만으로는 순위가 안 보일 수 있다 — 검증은 5절의 4 SE 판정과 4.7의 전 구간 곡선이 맡는다.

### 4.7 모든 $\lambda$에서의 순위 (단계 5)

$\lambda\in[0.1,1.9]$ 격자에서 두 설계의 $W,W_q$ 곡선을 그린다. 정리 4의 두 부등식이 전 구간에서 성립하는지 `assert`로 확인하고, 비율이 $2/(1+\rho)$, $2\rho/(1+\rho)$와 맞는지도 본다.

In [ ]:
lam_grid = np.linspace(0.1, 1.9, 37)
W1_curve = np.array([mmc_metrics(l, 2.0, 1)["W"] for l in lam_grid]); Wq1_curve = np.array([mmc_metrics(l, 2.0, 1)["Wq"] for l in lam_grid])
W2_curve = np.array([mmc_metrics(l, 1.0, 2)["W"] for l in lam_grid]); Wq2_curve = np.array([mmc_metrics(l, 1.0, 2)["Wq"] for l in lam_grid])
assert (W1_curve < W2_curve).all() and (Wq2_curve < Wq1_curve).all()          # 정리 4
rho_grid = lam_grid / 2
np.testing.assert_allclose(W2_curve / W1_curve, 2 / (1 + rho_grid), rtol=1e-10)
np.testing.assert_allclose(Wq2_curve / Wq1_curve, 2 * rho_grid / (1 + rho_grid), rtol=1e-10)

fig, ax = plt.subplots()
ax.semilogy(lam_grid, W1_curve, "C0-", label="W, one fast server (mu = 2)")
ax.semilogy(lam_grid, W2_curve, "C1-", label="W, two slow servers (mu = 1)")
ax.semilogy(lam_grid, Wq1_curve, "C0--", label="Wq, one fast server")
ax.semilogy(lam_grid, Wq2_curve, "C1--", label="Wq, two slow servers")
ax.axvline(LAM2, color="k", ls=":", lw=1); ax.annotate("lambda = 1.5 (Fig 5)", (LAM2, 0.02), xytext=(-95, 0), textcoords="offset points")
ax.set_xlabel("arrival rate lambda (total capacity 2 mu = 2)"); ax.set_ylabel("time (log scale)")
ax.set_title("Fig 6: W and Wq vs lambda: one fast server vs two slow servers")
ax.legend(loc="upper left")
plt.show()

실선($W$)은 파랑(빠른 1대)이 항상 아래, 점선($W_q$)은 주황(느린 2대)이 항상 아래다. $\lambda\to0$에서 $W$의 차이는 서비스 시간 차이 $1/\mu-1/(2\mu)$뿐이고, $W_q$는 느린 2대 쪽이 훨씬 작다(두 대가 모두 바쁠 확률 $\approx2\rho^2$ vs 한 대가 바쁠 확률 $\rho$). $\lambda\to2$에서는 둘 다 폭발하며 비율이 1로 수렴한다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
rows = [
    {"name": "P(wait), M/M/3 = Erlang C (step 2)", "predicted": predictions["p_wait_c3"], "estimate": p_wait_hat, "se": se_pwait, "exact": m3["p_wait"]},
    {"name": "Wq, M/M/3 (step 2)", "predicted": predictions["Wq_c3"], "estimate": Wq_hat, "se": se_Wq, "exact": m3["Wq"]},
    {"name": "W, M/M/3 (step 2)", "predicted": None, "estimate": W_hat, "se": se_W, "exact": m3["W"]},
    {"name": "E[Wq | Wq>0], M/M/3 (step 2)", "predicted": None, "estimate": cond_wait_hat, "se": se_cond, "exact": cond_exact},
    {"name": "P(block), M/M/3/3 = Erlang B (step 3)", "predicted": predictions["erlang_b_3_2"], "estimate": block_333, "se": se_b333, "exact": pi_333[3]},
    {"name": "P(block), M/M/2/5 = pi_5 (step 3)", "predicted": predictions["block_mm2_5"], "estimate": block_25, "se": se_b25, "exact": pi_mm25[5]},
    {"name": "W of admitted, M/M/2/5 = L/lam_eff (step 3)", "predicted": None, "estimate": W_25, "se": se_W25, "exact": W_25_exact},
    {"name": "W, one fast server (step 4)", "predicted": predictions["W_one_fast"], "estimate": W_fast, "se": se_Wf, "exact": m_fast["W"]},
    {"name": "Wq, one fast server (step 4)", "predicted": None, "estimate": Wq_fast, "se": se_Wqf, "exact": m_fast["Wq"]},
    {"name": "W, two slow servers (step 4)", "predicted": predictions["W_two_slow"], "estimate": W_two, "se": se_Wt, "exact": m_two["W"]},
    {"name": "Wq, two slow servers (step 4)", "predicted": None, "estimate": Wq_two, "se": se_Wqt, "exact": m_two["Wq"]},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **P(wait), M/M/3**: PASTA 또는 정리 1의 $C(c,a)$가 틀렸거나, 힙 배정이 FIFO 공용 대기열을 구현하지 못한 것(트랩 6).
- **Wq, W (M/M/3)**: $L_q=C\rho/(1-\rho)$ 또는 리틀 $W_q=L_q/\lambda$가 어긋난 것; P(wait)은 맞는데 이 둘만 어긋나면 워밍업 부족이나 SE 과소(배치 수)를 의심한다.
- **E[Wq | Wq>0]**: 평균은 맞는데 이 행이 어긋나면 대기시간의 **분포**가 $\mathrm{Exp}(c\mu-\lambda)$가 아니라는 뜻(FIFO가 아닌 규칙, 또는 지수가 아닌 서비스 — E3).
- **P(block), M/M/3/3**: Erlang B 공식 또는 $K$ 검사 힙(`in_system`에서 떠난 손님을 빼는 순서)이 잘못된 것.
- **P(block), M/M/2/5**: 절단 곱 형태 $\pi_K$나 PASTA(도착이 보는 $K$의 비율 = 시간 비율)가 어긋난 것.
- **W of admitted, M/M/2/5**: $L/\lambda$를 쓰면 $30/22=1.36$으로 어긋난다 — 이 행은 리틀에 $\lambda_{\rm eff}$를 써야 한다는 트랩 3의 검증이다.
- **W, Wq (fast / two)**: 정리 4의 손 계산 또는 `mmc_metrics`가 틀린 것. 두 시스템 모두 $\rho=0.75$라 오차막대가 크니, 4 SE 밖이면 먼저 배치 SE와 워밍업을 의심한다.

In [ ]:
assert_close(p_wait_hat, 4 / 9, se=se_pwait, k=4, name="p_wait_c3")
assert_close(Wq_hat, 4 / 9, se=se_Wq, k=4, name="Wq_c3")
assert_close(W_hat, 13 / 9, se=se_W, k=4, name="W_c3")
assert_close(cond_wait_hat, 1.0, se=se_cond, k=4, name="cond_wait_c3")
assert_close(block_333, 4 / 19, se=se_b333, k=4, name="erlang_b_3_2")
assert_close(block_25, 2 / 11, se=se_b25, k=4, name="block_mm2_5")
assert_close(W_25, 5 / 3, se=se_W25, k=4, name="W_mm2_5")
assert_close(W_fast, 2.0, se=se_Wf, k=4, name="W_one_fast"); assert_close(Wq_fast, 1.5, se=se_Wqf, k=4, name="Wq_one_fast")
assert_close(W_two, 16 / 7, se=se_Wt, k=4, name="W_two_slow"); assert_close(Wq_two, 9 / 7, se=se_Wqt, k=4, name="Wq_two_slow")
print("all Monte Carlo checks passed (4 SE, batch-means SE)")

## 6. 연습문제

### E1 계산

손으로: 창구 2개, $\lambda=1,\mu=1$ ($a=1$). $\pi_0$, Erlang C, $L_q,W_q,W,L$을 구하라. 같은 창구 2개를 손실 시스템(대기 불가)으로 바꾸면 차단 확률 $B(2,1)$은? 그리고 창구를 $c=1$로 줄이면 어떻게 되나?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$c=2$, $a=1$, $\rho=a/c=1/2$. 정리 1: $\pi_0^{-1}=\sum_{n<2}\frac{a^n}{n!}+\frac{a^2}{2!(1-\rho)}=1+1+\frac{1}{2\cdot0.5}=3$, $\pi_0=\frac13$.
$C=\pi_0\frac{a^2}{2!(1-\rho)}=\frac13\cdot1=\frac13$, $L_q=C\frac{\rho}{1-\rho}=\frac13\cdot1=\frac13$, $W_q=L_q/\lambda=\frac13$, $W=W_q+\frac1\mu=\frac43$, $L=L_q+a=\frac43$ (리틀 $\lambda W=\frac43$ ✓).

손실 시스템: $B(2,1)=\frac{1/2}{1+1+1/2}=\frac{1}{5}=0.2$ — 손님의 20%가 쫓겨난다.

$c=1$: $a=1=c$이므로 $\rho=1$ — M/M/1은 **불안정**(정상분포 없음, `mmc_metrics(1, 1, 1)`은 `ValueError`). 손실 시스템 M/M/1/1은 여전히 잘 정의되고 $B(1,1)=\frac{1}{1+1}=\frac12$: 절반이 차단된다. 유한 용량은 과부하에서도 정상분포를 가진다(정리 2).

```python
m = mmc_metrics(1.0, 1.0, 2)
print({k: round(v, 6) for k, v in m.items()})           # p_wait 1/3, Lq 1/3, Wq 1/3, W 4/3, L 4/3
print(erlang_b_recursive(2, 1.0), erlang_b_recursive(1, 1.0))   # 0.2, 0.5
try:
    mmc_metrics(1.0, 1.0, 1)
except ValueError as e:
    print("c=1:", e)                                      # unstable queue: rho = 1
```

</details>

### E2 유도 후 시뮬레이션 검증

$P(W_q>t)=C(c,a)\,e^{-(c\mu-\lambda)t}$를 유도하고(정리 3), M/M/3 ($\lambda=2,\mu=1$), $t=1$에서 4.4의 시뮬레이션 자료로 `(Wqw > 1).mean()` ± 배치 SE와 비교하라(`assert_close`, k=4). 또 $E[W_q]=\int_0^\infty P(W_q>t)\,dt=C/(c\mu-\lambda)$가 $L_q/\lambda$와 같음을 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** PASTA로 도착 손님이 $N=c+m$ ($m\ge0$)을 볼 확률은 $\pi_{c+m}=C(1-\rho)\rho^m$. 이때 서버 $c$대가 모두 바쁘고, FIFO라 내 앞의 $m$명이 서비스에 들어간 뒤 한 번 더 완료되어야 하므로 완료 $m+1$번을 기다린다. 대기 중 완료 간격은 i.i.d. $\mathrm{Exp}(c\mu)$(경쟁하는 시계 + 무기억성)이므로 조건부 대기는 $\mathrm{Erlang}(m+1,c\mu)$. 라플라스 변환을 기하 가중으로 섞으면 $E[e^{-sW_q};W_q>0]=C\frac{c\mu(1-\rho)}{c\mu(1-\rho)+s}$, $c\mu(1-\rho)=c\mu-\lambda$. 즉 $W_q\mid W_q>0\sim\mathrm{Exp}(c\mu-\lambda)$, $P(W_q>t)=Ce^{-(c\mu-\lambda)t}$.

$t=1$, $c\mu-\lambda=1$: $P(W_q>1)=\frac49e^{-1}=0.1635$.

$E[W_q]=\int_0^\infty Ce^{-(c\mu-\lambda)t}dt=\frac{C}{c\mu-\lambda}=\frac{4/9}{1}=\frac49$, 그리고 $L_q/\lambda=\frac{8/9}{2}=\frac49$ ✓ — 정리 3(분포)과 정리 1 + 리틀(평균)이 같은 답을 준다.

```python
t0 = 1.0
p1_exact = m3["p_wait"] * np.exp(-(C * MU - LAM) * t0)          # (4/9) e^{-1} = 0.163502
ind = (Wqw > t0).astype(float)
p1_hat, se_p1 = ind.mean(), batch_means_se(ind, 20)
print(f"P(Wq > 1): sim {p1_hat:.4f} ± {se_p1:.4f}, exact {p1_exact:.4f}")
assert_close(p1_hat, p1_exact, se=se_p1, k=4, name="P(Wq>1)")
print(f"C/(c mu - lambda) = {m3['p_wait'] / (C * MU - LAM):.6f} == Lq/lambda = {m3['Lq'] / LAM:.6f}")
```

</details>

### E3 가정을 깨보기

서비스 시간을 $\mathrm{Exp}(\mu)$ 대신 **상수** $1/\mu=1$로 바꿔(`service=lambda n, rng: np.ones(n)`) (a) M/D/3 ($\lambda=2$)의 대기 확률과 $W_q$, (b) M/D/3/3의 차단 비율을 시뮬레이션하라. Erlang C와 Erlang B 중 어느 것이 살아남는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) **M/D/3**: $P(\text{wait})\approx0.427$, $W_q\approx0.231$ — Erlang C의 $4/9=0.444$, $W_q=4/9=0.444$와 다르다. 대기 확률은 조금 낮고 $W_q$는 **약 절반**이다(M/D/c의 $W_q\approx\frac12\times$ M/M/c라는 경험칙 — 서비스 시간의 분산이 0이라 대기열에 누적되는 변동이 절반). 100만 명 참조값: $P(\text{wait})=0.4273\pm0.0008$, $W_q=0.2308\pm0.0008$.

(b) **M/D/3/3**: 차단 비율 $\approx0.2105=4/19$ — Erlang B는 그대로다(정리 2의 무감각성; 30만 명 참조값 $0.2104\pm0.0010$).

깨진 가정은 '지수 서비스'이고, **대기열이 있는** 시스템의 지표(Erlang C, $W_q$, 정리 3의 지수 조건부 대기)만 영향을 받는다. 손실 시스템에는 분산을 누적할 대기열이 없어 서비스 분포의 평균만 남는다. `assert_close`로 (b)는 통과하고 (a)는 4 SE를 넘어 실패해야 한다(그것이 '민감함'의 증거다).

```python
det = lambda k, rng: np.ones(k)                                   # 상수 서비스 1/mu = 1
Ad, sd, Dd = mmck_customers(LAM, MU, 3, N_CUST, rng, service=det)
Wqd = (sd - Ad)[warm:]
p_wait_d, se_pd = (Wqd > 0).mean(), batch_means_se((Wqd > 0).astype(float), 20)
Wq_d, se_Wqd = Wqd.mean(), batch_means_se(Wqd, 20)
print(f"(a) M/D/3: P(wait) = {p_wait_d:.4f} ± {se_pd:.4f} (Erlang C 0.4444),  Wq = {Wq_d:.4f} ± {se_Wqd:.4f} (M/M/3: 0.4444) -> ratio {Wq_d / m3['Wq']:.2f}")
Ab, sb, Db = mmck_customers(LAM, MU, 3, N_CUST, rng, K=3, service=det)
bd = np.isnan(Db)[warm:].astype(float)
print(f"(b) M/D/3/3: blocked = {bd.mean():.4f} ± {batch_means_se(bd, 20):.4f} (Erlang B 4/19 = {4/19:.4f})")
assert_close(bd.mean(), 4 / 19, se=batch_means_se(bd, 20), k=4, name="M/D/3/3 block")   # 통과 (무감각)
print("Wq deviation in SE units:", abs(Wq_d - m3["Wq"]) / se_Wqd)                        # >> 4 (민감)
```

</details>

## 7. 정리

1. M/M/c: $\pi_n\propto a^n/n!$ ($n\le c$), $\propto\frac{a^c}{c!}\rho^{n-c}$ ($n\ge c$), $\rho=a/c$; Erlang C $=P(N\ge c)$ = 대기 확률, $L_q=C\rho/(1-\rho)$, $L=L_q+a$, 나머지는 리틀.
2. M/M/c/K는 같은 곱 형태를 절단·재정규화한 것; 차단 확률 $\pi_K$(PASTA), 리틀은 $\lambda_{\rm eff}=\lambda(1-\pi_K)$로. $K=c$면 Erlang B(절단 푸아송) — 서비스 분포에 무감각.
3. 대기 조건부 대기시간은 $\mathrm{Exp}(c\mu-\lambda)$: $P(W_q>t)=Ce^{-(c\mu-\lambda)t}$, 평균 $C/(c\mu-\lambda)=L_q/\lambda$.
4. 같은 총 용량이면 빠른 1대가 $W$에서, 느린 2대가 $W_q$에서 이긴다(비율 $2/(1+\rho)$, $2\rho/(1+\rho)$) — 지표를 먼저 정하고 설계한다.
5. 다중 서버 시뮬레이션은 '가장 먼저 비는 서버' 힙으로; 유한 용량은 퇴장 시각 힙으로; SE는 배치 평균으로.

**trap 카드**
- Q: 서비스 시간 분포를 지수에서 다른 분포(같은 평균)로 바꾸면 Erlang B와 Erlang C 중 무엇이 그대로인가?
- A: Erlang B(손실 시스템 M/G/c/c)는 그대로다 — 무감각성. Erlang C와 $W_q$는 바뀐다(결정론적 서비스면 대기가 약 절반). 대기열이 서비스 시간의 분산을 누적하기 때문이다.

**다음 노트북**: 03f (선택) — $P(t)$를 `expm` 없이 푸아송 가중 DTMC 거듭제곱으로 얻는 균일화. CTMC 시뮬레이션의 두 번째 방법이자 'CTMC에는 주기가 없다'는 사실의 증명이다. 건너뛰면 03r로.

log/progress.md 한 줄 템플릿:
`- [ ] 03e M/M/c와 유한 용량 — 날짜: ____ / 예측 적중: __/6 / E1 __ E2 __ E3 __ / 메모: ____`